# 04c — Sensitivity of results to feature-policy choices

## Purpose
Notebook 03 froze one baseline feature policy of 21 inputs. Three of those inputs, and one structural assumption, are open to challenge:

| Choice | Concern |
|---|---|
| `verification_status` | It may not be available before an approval decision, and it is partly set by the lender in response to perceived risk (verified loans charge off *more*, not less). |
| `home_ownership` | It is a socioeconomic variable, and Notebook 06 uses it as the group for the fairness audit. Auditing a feature the model uses directly is not the same as auditing a proxy. |
| `addr_state` | It is a geographic variable and a possible proxy for protected characteristics. |
| 2014 labels used before 2015 | In production, 36-month loans issued in 2014 would not have final outcomes by January 2015. |

This notebook does **not** replace the baseline and does not alter any artifact. Each variant below gets its own feature list, its own freshly fitted preprocessor and its own models, built on top of `src/features/credit_features.py`. Nothing is saved to `models/checkpoints/`.

## What is compared
1. **Model quality** on 2014 and 2015 for each variant (AUC, Gini, Brier, mean prediction against observed rate).
2. **Decision and fairness outcomes** at a matched approval volume. For each variant the approval cut-off is the score quantile that approves the same share of **2014** loans as the frozen Notebook 05 policy, and is then applied unchanged to 2015. This uses development data only, and it puts every variant on the same footing for realized loss and group outcomes.

## Reference variants (not deployable)
* `pricing_reference` adds Lending Club's grade, sub-grade, interest rate and instalment (the Notebook 03 pricing policy). It shows how much of the lender's own pricing information the baseline leaves out.
* `grade_only_reference` uses grade and sub-grade alone. It is the benchmark any application-data model should be compared against.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

from src.features.credit_features import (
    BASELINE_FEATURES,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    PRICING_CATEGORICAL_FEATURES,
    PRICING_FEATURES,
    PRICING_NUMERIC_FEATURES,
    make_row_features,
    split_by_issue_month,
)

# When True, the refitted baseline must reproduce Notebook 04's 2014 AUC.
STRICT_REFERENCE_CHECKS = True

tables_dir = project_root / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

pd_file = project_root / "data" / "processed" / "pd_cohort_2011_2015_36m.parquet"
loss_file = project_root / "data" / "processed" / "loss_outcomes_2011_2015_36m.parquet"
policy_file = (
    project_root / "models" / "checkpoints" / "pd_policy_2014_unweighted_isotonic.json"
)

for path in (pd_file, loss_file, policy_file):
    if not path.is_file():
        raise FileNotFoundError(path)

print("Setup complete.")

Setup complete.


In [2]:
data = pd.read_parquet(pd_file)
splits = split_by_issue_month(data)

train_ready = make_row_features(splits["train"])
validation_ready = make_row_features(splits["validation"])
test_ready = make_row_features(splits["test"])

with policy_file.open("r", encoding="utf-8") as stream:
    policy = json.load(stream)

# Approval share on 2014 under the frozen Notebook 05 policy.
approval_share = policy["approved_development_loans"] / len(validation_ready)
lgd_assumption = float(policy["lgd_assumption"])

# Repayment fields, used only to evaluate decisions after they are made.
loss = pd.read_parquet(
    loss_file, columns=["id", "funded_amnt", "total_rec_prncp", "recoveries"]
)
loss["id"] = loss["id"].astype("string")
test_ready["id"] = test_ready["id"].astype("string")

test_eval = test_ready[["id", "charged_off", "home_ownership"]].merge(
    loss, on="id", how="left", validate="one_to_one"
)
assert test_eval["funded_amnt"].notna().all()
assert len(test_eval) == len(test_ready)

principal_loss_fraction = (
    1 - (test_eval["total_rec_prncp"] + test_eval["recoveries"]) / test_eval["funded_amnt"]
).clip(lower=0, upper=1)
test_eval["realized_loss"] = (
    test_eval["charged_off"] * principal_loss_fraction * test_eval["funded_amnt"]
)
test_eval["is_renter"] = (
    test_eval["home_ownership"].astype("string").str.upper().eq("RENT").fillna(False).astype(bool)
)

print(f"Frozen 2014 approval share (from Notebook 05 policy): {approval_share:.4f}")
print(f"Test loans: {len(test_eval):,}; renters: {int(test_eval['is_renter'].sum()):,}")

Frozen 2014 approval share (from Notebook 05 policy): 0.6780
Test loans: 283,026; renters: 121,463


## Variant definitions
Variants differ only in their feature list and, for one of them, the training window. Every variant uses the same LightGBM settings as Notebook 04 (200 trees, learning rate 0.05, `random_state=42`, unweighted) and the same logistic-regression settings.

`delayed_labels_2011_2012` trains on 2011–2012 loans only. It is a rough stand-in for the label-availability limitation: it uses an older, smaller training window to show how much model quality and drift change when the most recent labels are not available.

In [3]:
VERIFICATION = "verification_status"
HOME = "home_ownership"
GEO = "addr_state"


def without(features, *drop):
    return [f for f in features if f not in drop]


VARIANTS = {
    "baseline": {
        "features": list(BASELINE_FEATURES),
        "train_end": "2013-12-01",
        "note": "Frozen Notebook 03 policy",
    },
    "no_verification_status": {
        "features": without(BASELINE_FEATURES, VERIFICATION),
        "train_end": "2013-12-01",
        "note": "Drops verification_status",
    },
    "no_home_ownership": {
        "features": without(BASELINE_FEATURES, HOME),
        "train_end": "2013-12-01",
        "note": "Drops home_ownership",
    },
    "no_addr_state": {
        "features": without(BASELINE_FEATURES, GEO),
        "train_end": "2013-12-01",
        "note": "Drops addr_state",
    },
    "no_proxy_trio": {
        "features": without(BASELINE_FEATURES, VERIFICATION, HOME, GEO),
        "train_end": "2013-12-01",
        "note": "Drops all three",
    },
    "delayed_labels_2011_2012": {
        "features": list(BASELINE_FEATURES),
        "train_end": "2012-12-01",
        "note": "Baseline features, trained on 2011-2012 only",
    },
    "pricing_reference": {
        "features": list(PRICING_FEATURES),
        "train_end": "2013-12-01",
        "note": "Adds grade, sub_grade, int_rate, installment (reference only)",
    },
    "grade_only_reference": {
        "features": ["grade", "sub_grade"],
        "train_end": "2013-12-01",
        "note": "Lender grade alone (reference only)",
    },
}

NUMERIC_POOL = set(NUMERIC_FEATURES) | set(PRICING_NUMERIC_FEATURES)
CATEGORICAL_POOL = set(CATEGORICAL_FEATURES) | set(PRICING_CATEGORICAL_FEATURES)

for name, spec in VARIANTS.items():
    features = spec["features"]
    assert len(features) == len(set(features)), name
    assert set(features) <= NUMERIC_POOL | CATEGORICAL_POOL, name

summary = pd.DataFrame(
    {
        "features": {n: len(s["features"]) for n, s in VARIANTS.items()},
        "train_through": {n: s["train_end"][:7] for n, s in VARIANTS.items()},
        "note": {n: s["note"] for n, s in VARIANTS.items()},
    }
)
display(summary)

,features,train_through,note
baseline,21,2013-12,Frozen Notebook 03 policy
no_verification_status,20,2013-12,Drops verification_status
no_home_ownership,20,2013-12,Drops home_ownership
no_addr_state,20,2013-12,Drops addr_state
no_proxy_trio,18,2013-12,Drops all three
delayed_labels_2011_2012,21,2012-12,"Baseline features, trained on 2011-2012 only"
pricing_reference,25,2013-12,"Adds grade, sub_grade, int_rate, installment (..."
grade_only_reference,2,2013-12,Lender grade alone (reference only)


## Fit every variant
For each variant the preprocessor is rebuilt from the same recipe as Notebook 03 (median imputation and standardisation for numeric inputs, one-hot encoding with `handle_unknown="ignore"` for categorical inputs) and **fitted on that variant's training window only**.

In [4]:
def build_preprocessor(features):
    numeric = [f for f in features if f in NUMERIC_POOL]
    categorical = [f for f in features if f in CATEGORICAL_POOL]
    assert set(numeric) | set(categorical) == set(features)

    transformers = []
    if numeric:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric,
            )
        )
    if categorical:
        transformers.append(
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
        )
    return ColumnTransformer(transformers, remainder="drop")


def gini(auc):
    return 2 * auc - 1


results = []
fitted = {}

y_val = validation_ready["charged_off"].to_numpy()
y_test = test_ready["charged_off"].to_numpy()

for name, spec in VARIANTS.items():
    features = spec["features"]
    train_frame = train_ready.loc[train_ready["issue_month"] <= spec["train_end"]]
    y_train = train_frame["charged_off"].to_numpy()

    preprocessor = build_preprocessor(features)
    X_train = preprocessor.fit_transform(train_frame[features])
    X_val = preprocessor.transform(validation_ready[features])
    X_test = preprocessor.transform(test_ready[features])

    models = {
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        ),
        "Logistic regression": LogisticRegression(max_iter=1000, random_state=42),
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        p_val = model.predict_proba(X_val)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]

        for period, y, p in [("2014", y_val, p_val), ("2015", y_test, p_test)]:
            auc = roc_auc_score(y, p)
            results.append(
                {
                    "variant": name,
                    "model": model_name,
                    "period": period,
                    "train_loans": len(train_frame),
                    "auc": auc,
                    "gini": gini(auc),
                    "brier": brier_score_loss(y, p),
                    "mean_predicted": p.mean(),
                    "observed": y.mean(),
                    "observed_over_predicted": y.mean() / p.mean(),
                }
            )

        if model_name == "LightGBM":
            fitted[name] = {"val": p_val, "test": p_test}

    print(f"Fitted {name}")

quality = pd.DataFrame(results)

c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitted baseline


c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitted no_verification_status


c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitted no_home_ownership


... (6 outputs left out)

c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitted pricing_reference


c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitted grade_only_reference


## Check: the refitted baseline reproduces Notebook 04
The baseline variant was refitted from scratch here. Its 2014 AUC should match the numbers Notebook 04 printed (0.6600 for LightGBM, 0.6551 for logistic regression). A mismatch means the data or the feature module differs from the one used for the saved models, and nothing below should be trusted until it is resolved.

In [5]:
baseline_2014 = quality[(quality["variant"] == "baseline") & (quality["period"] == "2014")].set_index("model")["auc"]
expected = {"LightGBM": 0.6600, "Logistic regression": 0.6551}

for model_name, target in expected.items():
    observed = baseline_2014[model_name]
    ok = abs(observed - target) <= 1.5e-3
    print(f"Baseline {model_name} 2014 AUC: {observed:.4f} (Notebook 04: {target:.4f}) -> {'OK' if ok else 'MISMATCH'}")
    if STRICT_REFERENCE_CHECKS and not ok:
        raise AssertionError(f"Baseline {model_name} does not reproduce Notebook 04.")

Baseline LightGBM 2014 AUC: 0.6600 (Notebook 04: 0.6600) -> OK
Baseline Logistic regression 2014 AUC: 0.6551 (Notebook 04: 0.6551) -> OK


## Model quality by variant
Read the **AUC** columns for ranking quality, `observed_over_predicted` for level calibration (raw model output, no recalibration), and `brier` for probability accuracy. Differences of a few thousandths in AUC are small compared with the sampling noise shown in Notebook 04b.

In [6]:
wide = quality.pivot_table(
    index=["variant", "model"],
    columns="period",
    values=["auc", "brier", "observed_over_predicted"],
)
wide.columns = [f"{metric}_{period}" for metric, period in wide.columns]

order = list(VARIANTS)
wide = wide.reset_index()
wide["variant"] = pd.Categorical(wide["variant"], categories=order, ordered=True)
wide = wide.sort_values(["model", "variant"]).set_index(["model", "variant"])

display(wide[["auc_2014", "auc_2015", "brier_2014", "brier_2015", "observed_over_predicted_2015"]].round(4))

auc_2014  auc_2015  brier_2014  \
model               variant                                                    
LightGBM            baseline                    0.6600    0.6625      0.1140   
                    no_verification_status      0.6604    0.6624      0.1140   
                    no_home_ownership           0.6576    0.6593      0.1142   
                    no_addr_state               0.6588    0.6617      0.1141   
                    no_proxy_trio               0.6558    0.6581      0.1143   
                    delayed_labels_2011_2012    0.6426    0.6459      0.1153   
                    pricing_reference           0.6768    0.6831      0.1133   
                    grade_only_reference        0.6593    0.6784      0.1142   
Logistic regression baseline                    0.6551    0.6569      0.1144   
                    no_verification_status      0.6553    0.6577      0.1144   
                    no_home_ownership           0.6530    0.6538      0.1145   
                    no_addr_state               0.6535    0.6554      0.1145   
                    no_proxy_trio               0.6507    0.6524      0.1146   
                    delayed_labels_2011_2012    0.6424    0.6427      0.1153   
                    pricing_reference           0.6732    0.6798      0.1133   
                    grade_only_reference        0.6592    0.6783      0.1142   

                                              brier_2015  \
model               variant                                
LightGBM            baseline                      0.1219   
                    no_verification_status        0.1219   
                    no_home_ownership             0.1221   
                    no_addr_state                 0.1220   
                    no_proxy_trio                 0.1222   
                    delayed_labels_2011_2012      0.1228   
                    pricing_reference             0.1217   
                    grade_only_reference          0.1215   
Logistic regression baseline                      0.1223   
                    no_verification_status        0.1223   
                    no_home_ownership             0.1226   
                    no_addr_state                 0.1224   
                    no_proxy_trio                 0.1227   
                    delayed_labels_2011_2012      0.1230   
                    pricing_reference             0.1213   
                    grade_only_reference          0.1216   

                                              observed_over_predicted_2015  
model               variant                                                 
LightGBM            baseline                                        1.1570  
                    no_verification_status                          1.1605  
                    no_home_ownership                               1.1586  
                    no_addr_state                                   1.1548  
                    no_proxy_trio                                   1.1632  
                    delayed_labels_2011_2012                        1.0617  
                    pricing_reference                               1.3453  
                    grade_only_reference                            1.1874  
Logistic regression baseline                                        1.1477  
                    no_verification_status                          1.1603  
                    no_home_ownership                               1.1496  
                    no_addr_state                                   1.1485  
                    no_proxy_trio                                   1.1617  
                    delayed_labels_2011_2012                        1.0896  
                    pricing_reference                               1.2274  
                    grade_only_reference                            1.1878

## Decisions and group outcomes at matched approval volume
For each variant (LightGBM only):

1. Set the cut-off at the score quantile that approves the same share of **2014** loans as the frozen policy.
2. Apply that cut-off unchanged to **2015**.
3. Report the 2015 approval rate, the realized principal loss on approved loans (principal not recovered, as a share of approved funded amount), and the renter / non-renter outcomes.

Definitions: *rejection rate* is the share of a group rejected; *FPR* is the share of loans that did **not** charge off that were nonetheless rejected; *approval ratio* is the renter approval rate divided by the non-renter approval rate. Values below 0.80 are conventionally read as a warning sign under the four-fifths rule of thumb. This is a screening convention, and renters are a socioeconomic proxy, not a legally protected class.

Realized loss here is principal only. It ignores interest income, so it overstates how costly the loans being rejected really were. Net-return comparisons belong in Notebook 05 using the table from Notebook 01b.

In [7]:
def group_outcomes(frame, reject):
    y = frame["charged_off"].to_numpy()
    reject = np.asarray(reject, dtype=bool)
    negatives = y == 0
    return {
        "rejection_rate": reject.mean(),
        "fpr": reject[negatives].mean() if negatives.any() else np.nan,
        "approval_rate": 1 - reject.mean(),
    }


policy_rows = []
for name in VARIANTS:
    cutoff = np.quantile(fitted[name]["val"], approval_share)
    reject = fitted[name]["test"] > cutoff
    approved = ~reject

    approved_exposure = test_eval.loc[approved, "funded_amnt"].sum()
    realized_rate = test_eval.loc[approved, "realized_loss"].sum() / approved_exposure

    renter = test_eval["is_renter"].to_numpy()
    r = group_outcomes(test_eval[renter], reject[renter])
    n = group_outcomes(test_eval[~renter], reject[~renter])

    policy_rows.append(
        {
            "variant": name,
            "approval_rate_2015": approved.mean(),
            "realized_loss_rate_approved": realized_rate,
            "renter_rejection": r["rejection_rate"],
            "nonrenter_rejection": n["rejection_rate"],
            "approval_ratio_renter_vs_non": r["approval_rate"] / n["approval_rate"],
            "fpr_renter": r["fpr"],
            "fpr_nonrenter": n["fpr"],
            "fpr_ratio": r["fpr"] / n["fpr"],
        }
    )

policy_table = pd.DataFrame(policy_rows).set_index("variant")
display(policy_table.round(4))

base = policy_table.loc["baseline"]
print("Change relative to baseline (percentage points of realized loss rate, and change in FPR ratio):")
change = pd.DataFrame(
    {
        "realized_loss_rate_pp": (policy_table["realized_loss_rate_approved"] - base["realized_loss_rate_approved"]) * 100,
        "fpr_ratio_change": policy_table["fpr_ratio"] - base["fpr_ratio"],
        "approval_ratio_change": policy_table["approval_ratio_renter_vs_non"] - base["approval_ratio_renter_vs_non"],
    }
)
display(change.round(4))

,approval_rate_2015,realized_loss_rate_approved,renter_rejection,nonrenter_rejection,approval_ratio_renter_vs_non,fpr_renter,fpr_nonrenter,fpr_ratio
variant,,,,,,,,
baseline,0.6892,0.0527,0.4200,0.2288,0.7521,0.3850,0.2038,1.8894
no_verification_status,0.6891,0.0527,0.4207,0.2284,0.7508,0.3856,0.2037,1.8931
no_home_ownership,0.6887,0.0532,0.3822,0.2580,0.8327,0.3479,0.2323,1.4977
no_addr_state,0.6889,0.0524,0.4210,0.2285,0.7505,0.3861,0.2035,1.8976
no_proxy_trio,0.6884,0.0532,0.3767,0.2626,0.8453,0.3428,0.2365,1.4497
delayed_labels_2011_2012,0.6912,0.0548,0.3725,0.2609,0.8489,0.3409,0.2380,1.4325
pricing_reference,0.7355,0.0519,0.3425,0.2059,0.8279,0.3021,0.1788,1.6896
grade_only_reference,0.7552,0.0517,0.2792,0.2190,0.9229,0.2432,0.1917,1.2685


Change relative to baseline (percentage points of realized loss rate, and change in FPR ratio):


,realized_loss_rate_pp,fpr_ratio_change,approval_ratio_change
variant,,,
baseline,0.0000,0.0000,0.0000
no_verification_status,0.0072,0.0037,-0.0013
no_home_ownership,0.0500,-0.3917,0.0807
no_addr_state,-0.0242,0.0082,-0.0016
no_proxy_trio,0.0519,-0.4397,0.0933
delayed_labels_2011_2012,0.2097,-0.4569,0.0969
pricing_reference,-0.0716,-0.1998,0.0759
grade_only_reference,-0.0990,-0.6210,0.1708


## Save tables
Report tables only. No model, preprocessor, or processed-data file is written.

In [8]:
quality.to_csv(tables_dir / "04c_variant_model_quality.csv", index=False)
policy_table.reset_index().to_csv(tables_dir / "04c_variant_policy_outcomes.csv", index=False)
print("Saved: 04c_variant_model_quality.csv, 04c_variant_policy_outcomes.csv")

Saved: 04c_variant_model_quality.csv, 04c_variant_policy_outcomes.csv


## Limitations of this sensitivity analysis
* Removing a feature does not remove its information. Other inputs (income, FICO, loan purpose, DTI) can carry much of the same signal, so a small change in the group gap after removing `home_ownership` shows redundancy, not absence of a socioeconomic effect.
* The comparison is at a matched **approval volume**, not at the frozen probability threshold, because variants have different score scales. It is chosen on 2014 data only; 2015 outcomes play no part in the cut-off.
* `delayed_labels_2011_2012` is a rough stand-in for the label-availability problem. A proper treatment would use outcomes observed up to a decision date, which would also have to handle loans that had not yet resolved.
* Raw model scores are used, so level-calibration differences between variants are reported but not corrected here.
* Scores with few distinct values (the grade-only reference has about 35) cannot hit the target approval volume exactly, because loans with tied scores are approved or rejected together. Read `approval_rate_2015` before comparing that variant's loss rate with the others.
* All results describe originated 36-month loans from one historical window. They say nothing about applicants Lending Club rejected.
* The renter / non-renter split is a socioeconomic proxy. Meaningful fair-lending analysis would need protected-class data or an accepted proxy method, which this dataset does not provide.

## Conclusion and handoff

**AUC Changes:** Dropping `verification_status` or `addr_state` had almost zero impact on the 2015 AUC (0.6624 and 0.6617, respectively, vs. the 0.6625 baseline). Dropping `home_ownership` caused a very slight drop (0.6593). However, dropping the most recent training data (`delayed_labels_2011_2012`) severely degraded model quality, causing the 2015 AUC to plummet to 0.6459.

**Fairness Impacts:** Removing `home_ownership` proved highly effective at mitigating the socioeconomic disparity at a matched approval volume. It reduced the FPR ratio by 0.3917 (dropping it from 1.8894 to 1.4977) and improved the approval ratio by +0.0807 (lifting it from 0.7521 to 0.8327, crossing the 80% rule-of-thumb threshold).

**Grade-Only Comparison:** Lending Club's internal grade assessment captures more risk signal than our 21 application-data features combined. The `grade_only_reference` achieved a 2015 AUC of 0.6784, effortlessly beating the application-data baseline (0.6625). This highlights a severe limitation of using originated-loan datasets to simulate pre-approval models: the lender's hidden underwriting parameters are highly predictive.

**Delayed Labels:** Limiting the training window to 2011–2012 (to simulate a realistic delay in label maturity) caused the 2015 AUC to drop from 0.6625 to 0.6459. Furthermore, it severely degraded level-calibration (the observed_over_predicted ratio dropped to 1.0617, though this artificially masked under-prediction due to the overall lower AUC ranking quality).

**Recommended Baseline for Notebooks 05 and 06:** I will proceed with the **Frozen Notebook 03 Baseline**. While `no_home_ownership` creates a fairer policy, carrying the standard baseline forward allows us to accurately audit and demonstrate the natural socioeconomic biases, calibration flaws, and drift vulnerabilities inherent in standard credit risk pipelines in Notebooks 05 and 06.